In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

# --- Step 0: Load the Data ---
# Load the dataset from the specified file path
file_path = '/content/final_model_data.csv'
try:
    df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
except FileNotFoundError:
    print(f"Error: The file was not found at {file_path}")
    # Exit or handle the error appropriately
    exit()

# --- Step 1: Preparing Data ---
print("\n--- Step 1: Preparing Data ---")

# Convert the 'date' column to datetime and set it as the DataFrame index
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Create the target variable: 1 if water area is in the top 15%, 0 otherwise
# This defines a "flood event" for our classification task
flood_threshold = df['river_water_area_sqkm'].quantile(0.85)
df['Flood_Event'] = (df['river_water_area_sqkm'] > flood_threshold).astype(int)
print(f"Flood event threshold (river area > {flood_threshold:.2f} sqkm) has been defined.")
print("Distribution of Flood Events (1) vs. Non-Events (0):")
print(df['Flood_Event'].value_counts())

# Separate the columns into Geospatial (static) and Temporal (time-varying) sets
geospatial_cols = [col for col in df.columns if 'mean' in col or 'land_cover' in col]
temporal_cols = ['river_water_area_sqkm', 'rainfall_mm']

# Create the initial DataFrames for each data type
X_geo = df[geospatial_cols]
y_target = df['Flood_Event']
X_temporal_raw = df[temporal_cols]

# Normalize the temporal data for the neural network
# This is crucial for the performance of models like CNNs
scaler = StandardScaler()
X_temporal_scaled = scaler.fit_transform(X_temporal_raw)

# --- Step 2: Create Time-Series Sequences ---
# This function converts a continuous time series into overlapping windows (sequences),
# which is the required input format for time-series models (like CNN, LSTM, GRU).
def create_sequences(data, n_steps):
    """
    Prepares time-series data into supervised learning format.
    - data: The input time-series data (numpy array).
    - n_steps: The number of past time steps to use as input features.
    """
    X, y = [], []
    for i in range(len(data)):
        # Find the end of this pattern
        end_ix = i + n_steps
        # Check if we are beyond the dataset
        if end_ix > len(data)-1:
            break
        # Gather input and output parts of the pattern
        # The sequence is the input (X), the next value is the output (y)
        seq_x, seq_y = data[i:end_ix, :], data[end_ix, 0] # Predicting the next river_water_area
        X.append(seq_x)
        y.append(seq_y)
    return np.array(X), np.array(y)

# Define how many past months will be used to predict the next month
N_STEPS = 3

# Generate the temporal sequences and their corresponding forecast targets
X_temporal, y_temporal = create_sequences(X_temporal_scaled, N_STEPS)

# --- Step 3: Align Datasets ---
# Because we used N_STEPS to create sequences, the first few rows of the original
# data cannot be used. We must trim the geospatial and target datasets to match.
X_geo_aligned = X_geo.iloc[N_STEPS:]
y_target_aligned = y_target.iloc[N_STEPS:]

# --- Step 4: Final Verification ---
print(f"\nTemporal data has been reshaped into sequences using {N_STEPS} time steps.")
print("\nFinal shapes of the prepared datasets:")
print(f"Geospatial features (X_geo_aligned): {X_geo_aligned.shape}")
print(f"Temporal sequences (X_temporal):     {X_temporal.shape}")
print(f"Final Target (y_target_aligned):     {y_target_aligned.shape}")

# Verify that the number of samples is consistent across all datasets
assert len(X_geo_aligned) == len(X_temporal) == len(y_target_aligned)
print("\nVerification successful: All datasets are aligned and ready for model training.")

Successfully loaded data from /content/final_model_data.csv

--- Step 1: Preparing Data ---
Flood event threshold (river area > 1.23 sqkm) has been defined.
Distribution of Flood Events (1) vs. Non-Events (0):
Flood_Event
0    79
1    14
Name: count, dtype: int64

Temporal data has been reshaped into sequences using 3 time steps.

Final shapes of the prepared datasets:
Geospatial features (X_geo_aligned): (90, 9)
Temporal sequences (X_temporal):     (90, 3, 2)
Final Target (y_target_aligned):     (90,)

Verification successful: All datasets are aligned and ready for model training.


In [ ]:
# Import necessary libraries for modeling and evaluation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# --- Step 2: Train Baseline Classical Models ---
print("\n--- Step 2: Training Baseline Classical Models ---")

# We use the ALIGNED geospatial data for a fair comparison.
# The target variable (y_target_aligned) is also aligned.
# 'stratify' is used to ensure the train/test split has the same proportion of flood events as the original dataset.
X_train_geo, X_test_geo, y_train, y_test = train_test_split(
    X_geo_aligned, y_target_aligned, test_size=0.2, random_state=42, stratify=y_target_aligned
)

print(f"Data split into training ({len(X_train_geo)} samples) and testing ({len(X_test_geo)} samples) sets.")

# --- Train and Evaluate AdaBoost Classifier ---
print("\nTraining AdaBoost model...")
adaboost_model = AdaBoostClassifier(n_estimators=100, random_state=42)
adaboost_model.fit(X_train_geo, y_train)

# Make predictions on the test set
y_pred_adaboost = adaboost_model.predict(X_test_geo)
base_adaboost_acc = accuracy_score(y_test, y_pred_adaboost)
print(f"Baseline AdaBoost Accuracy: {base_adaboost_acc:.4f}")

# --- Train and Evaluate K-Nearest Neighbors (KNN) Classifier ---
print("\nTraining KNN model...")
# Using 3 neighbors is a common starting point
knn_model = KNeighborsClassifier(n_neighbors=3)
knn_model.fit(X_train_geo, y_train)

# Make predictions on the test set
y_pred_knn = knn_model.predict(X_test_geo)
base_knn_acc = accuracy_score(y_test, y_pred_knn)
print(f"Baseline KNN Accuracy: {base_knn_acc:.4f}")

# --- Display Detailed Reports ---
print("\n--- Detailed Classification Reports (Baseline) ---")
print("\nAdaBoost Classifier Report:")
print(classification_report(y_test, y_pred_adaboost, zero_division=0))

print("\nKNN Classifier Report:")
print(classification_report(y_test, y_pred_knn, zero_division=0))


--- Step 2: Training Baseline Classical Models ---
Data split into training (72 samples) and testing (18 samples) sets.

Training AdaBoost model...
Baseline AdaBoost Accuracy: 0.8333

Training KNN model...
Baseline KNN Accuracy: 0.8333

--- Detailed Classification Reports (Baseline) ---

AdaBoost Classifier Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        15
           1       0.00      0.00      0.00         3

    accuracy                           0.83        18
   macro avg       0.42      0.50      0.45        18
weighted avg       0.69      0.83      0.76        18


KNN Classifier Report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91        15
           1       0.00      0.00      0.00         3

    accuracy                           0.83        18
   macro avg       0.42      0.50      0.45        18
weighted avg       0.69      0.83      0.76        18



In [ ]:
# Import the necessary library for deep learning
import tensorflow as tf

# --- Step 3: Train the Deep Learning Model (1D-CNN) ---
print("\n--- Step 3: Training 1D-CNN for Feature Extraction ---")

# Get the number of features from the shape of our temporal data array
# Shape is (samples, timesteps, features), so we need the last dimension
N_FEATURES = X_temporal.shape[2]

# --- Define the CNN Architecture ---
# We use a simple, sequential model
cnn_model = tf.keras.models.Sequential([
    # Layer 1: The convolutional layer learns local patterns (like a spike in rainfall over 2 months)
    # 64 filters, kernel size of 2 (looks at 2 months at a time)
    tf.keras.layers.Conv1D(filters=64, kernel_size=2, activation='relu', input_shape=(N_STEPS, N_FEATURES)),

    # Layer 2: Max pooling reduces the complexity and highlights the most important features
    tf.keras.layers.MaxPooling1D(pool_size=2),

    # Layer 3: Flatten converts the 2D output of the conv/pool layers into a 1D vector
    tf.keras.layers.Flatten(),

    # Layer 4: This is our key feature-rich layer. We give it a name to easily access it later.
    tf.keras.layers.Dense(50, activation='relu', name='feature_layer'),

    # Layer 5: The final output layer for the forecasting task (predicting the next river_water_area)
    tf.keras.layers.Dense(1, name='output_layer')
])

# --- Compile the Model ---
# We configure the model for training with an optimizer and a loss function.
# 'adam' is a robust default optimizer. 'mean_squared_error' is standard for regression tasks.
cnn_model.compile(optimizer='adam', loss='mean_squared_error')

# Print a summary of the model's architecture
print("\nCNN Model Architecture:")
cnn_model.summary()

# --- Train the Model ---
# We fit the model to our temporal data.
# 'epochs' is the number of times the model will see the entire dataset.
# 'verbose=0' runs the training silently for a cleaner output.
print("\nTraining the CNN model on temporal sequences...")
cnn_model.fit(X_temporal, y_temporal, epochs=200, batch_size=4, verbose=0)
print("1D-CNN model trained successfully.")


--- Step 3: Training 1D-CNN for Feature Extraction ---

CNN Model Architecture:


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 2, 64)          │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 1, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feature_layer (Dense)           │ (None, 50)             │         3,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,621 (14.14 KB)

 Trainable params: 3,621 (14.14 KB)

 Non-trainable params: 0 (0.00 B)


Training the CNN model on temporal sequences...
1D-CNN model trained successfully.


In [ ]:
# Import necessary libraries (some may be redundant if running in one script)
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# --- Step 4: Create the Hybrid Models ---
print("\n--- Step 4: Creating Hybrid Models ---")

# Part A: Build the feature extractor from the trained CNN
feature_extractor_model = tf.keras.Model(
    inputs=cnn_model.inputs,
    outputs=cnn_model.get_layer(name='feature_layer').output,
    name="cnn_feature_extractor"
)

# Part B: Extract deep features and combine datasets
print("\nExtracting deep features from temporal data...")
temporal_features = feature_extractor_model.predict(X_temporal)
temporal_features_df = pd.DataFrame(temporal_features, index=X_geo_aligned.index)

X_hybrid = pd.concat([X_geo_aligned.reset_index(drop=True), temporal_features_df.reset_index(drop=True)], axis=1)

# ***************************************************************
# *** THE FIX: Convert all column names to strings to prevent the TypeError ***
X_hybrid.columns = X_hybrid.columns.astype(str)
# ***************************************************************

print("Hybrid dataset created and column names standardized.")
print(f"Shape of the final Hybrid Dataset: {X_hybrid.shape}")

# Part C: Train the final hybrid models on the new dataset
X_train_hybrid, X_test_hybrid, y_train_hybrid, y_test_hybrid = train_test_split(
    X_hybrid, y_target_aligned, test_size=0.2, random_state=42, stratify=y_target_aligned
)

# Train Hybrid AdaBoost
print("\nTraining Hybrid AdaBoost model...")
hybrid_adaboost_model = AdaBoostClassifier(n_estimators=100, random_state=42)
hybrid_adaboost_model.fit(X_train_hybrid, y_train_hybrid)
y_pred_hybrid_adaboost = hybrid_adaboost_model.predict(X_test_hybrid)
hybrid_adaboost_acc = accuracy_score(y_test_hybrid, y_pred_hybrid_adaboost)
print(f"Hybrid AdaBoost Accuracy: {hybrid_adaboost_acc:.4f}")

# Train Hybrid KNN
print("\nTraining Hybrid KNN model...")
hybrid_knn_model = KNeighborsClassifier(n_neighbors=3)
hybrid_knn_model.fit(X_train_hybrid, y_train_hybrid)
y_pred_hybrid_knn = hybrid_knn_model.predict(X_test_hybrid)
hybrid_knn_acc = accuracy_score(y_test_hybrid, y_pred_hybrid_knn)
print(f"Hybrid KNN Accuracy: {hybrid_knn_acc:.4f}")


# --- Step 5: Final Comparison and Conclusion ---
print("\n--- Step 5: Final Performance Comparison ---")
print("="*40)
print(f"Baseline AdaBoost Accuracy (Geo only): {base_adaboost_acc:.4f}")
print(f"Hybrid AdaBoost Accuracy (Geo + CNN):   {hybrid_adaboost_acc:.4f}")
print("\n")
print(f"Baseline KNN Accuracy (Geo only):      {base_knn_acc:.4f}")
print(f"Hybrid KNN Accuracy (Geo + CNN):        {hybrid_knn_acc:.4f}")
print("="*40)


--- Step 4: Creating Hybrid Models ---

Extracting deep features from temporal data...
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Hybrid dataset created and column names standardized.
Shape of the final Hybrid Dataset: (90, 59)

Training Hybrid AdaBoost model...
Hybrid AdaBoost Accuracy: 0.8333

Training Hybrid KNN model...
Hybrid KNN Accuracy: 0.8889

--- Step 5: Final Performance Comparison ---
Baseline AdaBoost Accuracy (Geo only): 0.8333
Hybrid AdaBoost Accuracy (Geo + CNN):   0.8333


Baseline KNN Accuracy (Geo only):      0.8333
Hybrid KNN Accuracy (Geo + CNN):        0.8889


In [ ]:
# --- Step 6: Analyze Feature Importance ---
print("\n--- Step 6: Analyzing Feature Importance for Hybrid AdaBoost ---")

# Get the feature importances from the trained hybrid AdaBoost model
importances = hybrid_adaboost_model.feature_importances_
feature_names = X_hybrid.columns

# Create a DataFrame for easy viewing and sorting
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
})

# Sort the features by importance in descending order
importance_df = importance_df.sort_values(by='importance', ascending=False)

print("Top 15 Most Important Features for Hybrid AdaBoost:")
print(importance_df.head(15))

# Check the importance of the CNN features
# The CNN feature columns are named '0', '1', '2', etc.
cnn_feature_importance = importance_df[importance_df['feature'].astype(str).str.isdigit()]['importance'].sum()

print(f"\nTotal Importance of all CNN-extracted features: {cnn_feature_importance:.4f}")


--- Step 6: Analyzing Feature Importance for Hybrid AdaBoost ---
Top 15 Most Important Features for Hybrid AdaBoost:
   feature  importance
20      11    0.125845
37      28    0.119218
27      18    0.113550
52      43    0.054893
30      21    0.054161
36      27    0.052630
23      14    0.045820
28      19    0.040519
47      38    0.033569
39      30    0.032687
11       2    0.032651
31      22    0.030321
19      10    0.027941
29      20    0.026935
54      45    0.024594

Total Importance of all CNN-extracted features: 1.0000


In [ ]:
# Import the necessary libraries for saving models
import joblib
import tensorflow as tf

# --- Step 1: Save all trained components ---

# 1. Save the StandardScaler
# This ensures new data is scaled exactly the same way as the training data
joblib.dump(scaler, 'temporal_scaler.gz')
print("StandardScaler saved to temporal_scaler.gz")

# 2. Save the CNN feature extractor model
# We save the entire model (architecture, weights, etc.)
feature_extractor_model.save('cnn_feature_extractor.h5')
print("CNN Feature Extractor saved to cnn_feature_extractor.h5")

# 3. Save the final Hybrid KNN model
joblib.dump(hybrid_knn_model, 'hybrid_knn_model.gz')
print("Hybrid KNN Model saved to hybrid_knn_model.gz")

StandardScaler saved to temporal_scaler.gz
CNN Feature Extractor saved to cnn_feature_extractor.h5
Hybrid KNN Model saved to hybrid_knn_model.gz


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import joblib
import tensorflow as tf

# --- Step 2: Load the saved pipeline components ---
# This part would typically be at the start of your application server
try:
    scaler = joblib.load('temporal_scaler.gz')
    feature_extractor = tf.keras.models.load_model('cnn_feature_extractor.h5')
    knn_classifier = joblib.load('hybrid_knn_model.gz')
    print("All pipeline components loaded successfully.")
except Exception as e:
    print(f"Error loading models: {e}")


# --- The Core Inference Function ---
def predict_flood_risk(geospatial_data, temporal_data):
    """
    Predicts flood risk using the full hybrid model pipeline.

    Args:
        geospatial_data (pd.DataFrame): A DataFrame with one row containing the
                                        static geospatial features.
        temporal_data (np.ndarray): A NumPy array of shape (3, 2) containing the
                                    last 3 months of [river_water_area, rainfall_mm].

    Returns:
        dict: A dictionary containing the prediction and a status message.
    """
    try:
        # 1. Scale the new temporal data using the loaded scaler
        temporal_data_scaled = scaler.transform(temporal_data)

        # 2. Reshape the data for the CNN: (samples, timesteps, features)
        temporal_sequence = np.expand_dims(temporal_data_scaled, axis=0)

        # 3. Extract deep features using the loaded CNN feature extractor
        deep_features = feature_extractor.predict(temporal_sequence)

        # 4. Combine geospatial features with the new deep features
        deep_features_df = pd.DataFrame(deep_features)

        # Reset indices to ensure a clean concatenation
        geospatial_data = geospatial_data.reset_index(drop=True)

        hybrid_features = pd.concat([geospatial_data, deep_features_df], axis=1)

        # 5. Standardize column names to strings to prevent errors
        hybrid_features.columns = hybrid_features.columns.astype(str)

        # 6. Make the final prediction using the loaded KNN model
        prediction = knn_classifier.predict(hybrid_features)[0]
        probability = knn_classifier.predict_proba(hybrid_features)[0]

        # 7. Return a clear, simple output
        if prediction == 1:
            return {
                "prediction": 1,
                "risk_level": "High",
                "message": f"Flood Warning: High risk of a flood event detected.",
                "confidence": f"{np.max(probability)*100:.2f}%"
            }
        else:
            return {
                "prediction": 0,
                "risk_level": "Low",
                "message": "No immediate flood risk detected.",
                "confidence": f"{np.max(probability)*100:.2f}%"
            }

    except Exception as e:
        return {"error": str(e)}

All pipeline components loaded successfully.


In [ ]:
# --- Step 3: Example Usage ---
# Now we will call the predict_flood_risk function with sample data
# to simulate a real-world prediction request.

# 1. Define the static geospatial features for a new, hypothetical location.
# The column names MUST exactly match the ones used during training.
new_geospatial_data = pd.DataFrame([{
    'mean_elevation_meters': 295.0,
    'mean_slope_degrees': 5.2,
    'land_cover_class_10_percent': 25.0,
    'land_cover_class_20_percent': 1.5,
    'land_cover_class_30_percent': 10.0,
    'land_cover_class_40_percent': 20.0,
    'land_cover_class_50_percent': 17.0,
    'land_cover_class_60_percent': 12.0,
    'land_cover_class_80_percent': 14.5
}])

# 2. Provide the temporal data for the LAST 3 months for that location.
# The shape must be (3, 2) -> [[month1_river, month1_rain], [month2...], [month3...]]

# Example 1: A low-risk scenario with normal conditions
low_risk_temporal_data = np.array([
    [0.6, 0.1],  # 3 months ago
    [0.5, 0.0],  # 2 months ago
    [0.7, 0.2]   # Last month
])

# Example 2: A high-risk scenario with rising water levels and heavy rain
high_risk_temporal_data = np.array([
    [1.0, 2.5],   # 3 months ago
    [1.5, 8.0],   # 2 months ago
    [1.8, 15.0]  # Last month (high values that would likely trigger a flood event)
])


# 3. Call the inference function and print the results
print("\n--- Making Predictions ---")

# Test the low-risk scenario
result_low = predict_flood_risk(new_geospatial_data, low_risk_temporal_data)
print(f"\nLow Risk Scenario Result:")
print(result_low)

# Test the high-risk scenario
result_high = predict_flood_risk(new_geospatial_data, high_risk_temporal_data)
print(f"\nHigh Risk Scenario Result:")
print(result_high)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(



--- Making Predictions ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

Low Risk Scenario Result:
{'prediction': 0, 'risk_level': 'Low', 'message': 'No immediate flood risk detected.', 'confidence': '100.00%'}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step

High Risk Scenario Result:
{'prediction': 0, 'risk_level': 'Low', 'message': 'No immediate flood risk detected.', 'confidence': '66.67%'}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [ ]:
from graphviz import Digraph

# Create a new Digraph object with a top-to-bottom layout and straight lines
dot = Digraph(
    comment='Simple Project Flowchart',
    graph_attr={
        'rankdir': 'TB',
        'splines': 'ortho'
    }
)
dot.attr('node', shape='box', style='filled')

# Phase 1: Data Preparation
dot.node('A', 'Start: Raw Scattered Files', fillcolor='lightblue')
dot.node('B', 'Combine & Aggregate Data\n(Using Memory-Safe Chunks)', fillcolor='white')
dot.node('C', 'Final Dataset:\nFINAL_multimodal_dataset.csv', shape='cylinder', fillcolor='yellow')

# Phase 2: Initial Modeling & Analysis
dot.node('D', 'Train Initial Models (ANN & CNN)', fillcolor='white')
dot.node('E', 'Result: Poor Performance\n(High RMSE)', shape='diamond', fillcolor='salmon')
dot.node('F', 'Debug: Find Feature Importance with XGBoost', fillcolor='white')
dot.node('G', 'Insight: Only 5 EEG Features Matter', shape='diamond', fillcolor='lightgreen')

# Phase 3: Model Optimization
dot.node('H', 'Retrain XGBoost on Top 5 Features', fillcolor='white')
dot.node('I', 'Apply Feature Engineering\n(Polynomial Features)', fillcolor='white')
dot.node('J', 'Final Best Model:\nXGBoost on Engineered Features\n(RMSE = 62.30)', shape='hexagon', fillcolor='gold')
dot.node('K', 'End: Save & Interpret Model', fillcolor='lightblue')

# Connect the nodes to show the flow
dot.edge('A', 'B')
dot.edge('B', 'C')
dot.edge('C', 'D')
dot.edge('D', 'E')
dot.edge('E', 'F')
dot.edge('F', 'G')
dot.edge('G', 'H')
dot.edge('H', 'I')
dot.edge('I', 'J')
dot.edge('J', 'K')

# Render the flowchart
dot.render('simple_project_flowchart', view=True, format='png')

print("Flowchart 'simple_project_flowchart.png' has been generated.")

Flowchart 'simple_project_flowchart.png' has been generated.
